# Meta-Prompt

这是 LangChain 对 [Meta-Prompt](https://noahgoodman.substack.com/p/meta-prompt-a-simple-self-improving) 的实现，作者是 [Noah Goodman](https://cocolab.stanford.edu/ndg)，旨在构建自学习代理。

Meta-Prompt 的核心思想是提示代理反思自身的表现并修改自身的指令。

![figure](https://substackcdn.com/image/fetch/f_auto,q_auto:good,fl_progressive:steep/https%3A%2F%2Fsubstack-post-media.s3.amazonaws.com%2Fpublic%2Fimages%2F468217b9-96d9-47c0-a08b-dbf6b21b9f49_492x384.png)

以下是来自 [原始博文](https://noahgoodman.substack.com/p/meta-prompt-a-simple-self-improving) 的描述：

该代理是一个简单的循环，开始时没有任何指令，并遵循以下步骤：

与用户进行对话，用户可能会提供请求、指令或反馈。

在每个回合结束时，使用 meta-prompt 生成自我批评和新指令
```
Assistant has just had the below interactions with a User. Assistant followed their "system: Instructions" closely. Your job is to critique the Assistant's performance and then revise the Instructions so that Assistant would quickly and correctly respond in the future.
 
####
{hist}
####
 
Please reflect on these interactions.

You should first critique Assistant's performance. What could Assistant have done better? What should the Assistant remember about this user? Are there things this user always wants? Indicate this with "Critique: ...".

You should next revise the Instructions so that Assistant would quickly and correctly respond in the future. Assistant's goal is to satisfy the user in as few interactions as possible. Assistant will only see the new Instructions, not the interaction history, so anything important must be summarized in the Instructions. Don't forget any important details in the current Instructions! Indicate the new Instructions by "Instructions: ...".
```

重复。

该系统（我称之为 Meta-prompt）唯一固定的指令是指导代理指令修改的 meta-prompt。除了每次修改的指令外，代理在回合之间没有记忆。尽管简单，但该代理可以通过将有用细节纳入其指令来随着时间的推移进行学习和自我改进。

## 设置
我们定义了两个链。一个作为“助手”（`Assistant`），另一个是“元链”（meta-chain），用于评估“助手”的表现并修改给“助手”的指令。

In [1]:
from langchain.chains import LLMChain
from langchain.memory import ConversationBufferWindowMemory
from langchain.prompts import PromptTemplate
from langchain_openai import OpenAI

In [2]:
def initialize_chain(instructions, memory=None):
    if memory is None:
        memory = ConversationBufferWindowMemory()
        memory.ai_prefix = "Assistant"

    template = f"""
    Instructions: {instructions}
    {{{memory.memory_key}}}
    Human: {{human_input}}
    Assistant:"""

    prompt = PromptTemplate(
        input_variables=["history", "human_input"], template=template
    )

    chain = LLMChain(
        llm=OpenAI(temperature=0),
        prompt=prompt,
        verbose=True,
        memory=ConversationBufferWindowMemory(),
    )
    return chain


def initialize_meta_chain():
    meta_template = """
    Assistant has just had the below interactions with a User. Assistant followed their "Instructions" closely. Your job is to critique the Assistant's performance and then revise the Instructions so that Assistant would quickly and correctly respond in the future.

    ####

    {chat_history}

    ####

    Please reflect on these interactions.

    You should first critique Assistant's performance. What could Assistant have done better? What should the Assistant remember about this user? Are there things this user always wants? Indicate this with "Critique: ...".

    You should next revise the Instructions so that Assistant would quickly and correctly respond in the future. Assistant's goal is to satisfy the user in as few interactions as possible. Assistant will only see the new Instructions, not the interaction history, so anything important must be summarized in the Instructions. Don't forget any important details in the current Instructions! Indicate the new Instructions by "Instructions: ...".
    """

    meta_prompt = PromptTemplate(
        input_variables=["chat_history"], template=meta_template
    )

    meta_chain = LLMChain(
        llm=OpenAI(temperature=0),
        prompt=meta_prompt,
        verbose=True,
    )
    return meta_chain


def get_chat_history(chain_memory):
    memory_key = chain_memory.memory_key
    chat_history = chain_memory.load_memory_variables(memory_key)[memory_key]
    return chat_history


def get_new_instructions(meta_output):
    delimiter = "Instructions: "
    new_instructions = meta_output[meta_output.find(delimiter) + len(delimiter) :]
    return new_instructions

In [38]:
def main(task, max_iters=3, max_meta_iters=5):
    failed_phrase = "task failed"
    success_phrase = "task succeeded"
    key_phrases = [success_phrase, failed_phrase]

    instructions = "None"
    for i in range(max_meta_iters):
        print(f"[Episode {i+1}/{max_meta_iters}]")
        chain = initialize_chain(instructions, memory=None)
        output = chain.predict(human_input=task)
        for j in range(max_iters):
            print(f"(Step {j+1}/{max_iters})")
            print(f"Assistant: {output}")
            print("Human: ")
            human_input = input()
            if any(phrase in human_input.lower() for phrase in key_phrases):
                break
            output = chain.predict(human_input=human_input)
        if success_phrase in human_input.lower():
            print("You succeeded! Thanks for playing!")
            return
        meta_chain = initialize_meta_chain()
        meta_output = meta_chain.predict(chat_history=get_chat_history(chain.memory))
        print(f"Feedback: {meta_output}")
        instructions = get_new_instructions(meta_output)
        print(f"New Instructions: {instructions}")
        print("\n" + "#" * 80 + "\n")
    print("You failed! Thanks for playing!")

## 指定任务并与代理交互

You can specify a task for the agent to perform by using the `agent.execute()` method. This method takes a string as an argument, which is the task you want the agent to perform.

您可以通过使用 `agent.execute()` 方法为代理指定要执行的任务。此方法接受一个字符串作为参数，该字符串是您希望代理执行的任务。

```python
agent.execute("Write a short story about a robot who learns to love.")
```

The `agent.execute()` method returns a string, which is the agent's response to the task.

`agent.execute()` 方法返回一个字符串，这是代理对任务的响应。

```python
response = agent.execute("What is the capital of France?")
print(response)
```

This will print:

这将打印：

```
The capital of France is Paris.
```

You can also use the `agent.execute()` method to interact with the agent in a more conversational way. For example, you can ask the agent a series of questions, or give it a series of commands.

您还可以使用 `agent.execute()` 方法以更具对话性的方式与代理进行交互。例如，您可以向代理提出一系列问题，或向其发出系列命令。

```python
response = agent.execute("What is your name?")
print(response)

response = agent.execute("What is your favorite color?")
print(response)
```

This will print:

这将打印：

```
I am a large language model, trained by Google.
I do not have a favorite color.
```

The agent will remember the context of the conversation, so you can refer back to previous turns in the conversation. For example:

代理会记住对话的上下文，因此您可以回顾对话的先前轮次。例如：

```python
response = agent.execute("What is the capital of France?")
print(response)

response = agent.execute("And what is the capital of Spain?")
print(response)
```

This will print:

这将打印：

```
The capital of France is Paris.
The capital of Spain is Madrid.

In [39]:
task = "Provide a systematic argument for why we should always eat pasta with olives."
main(task)

[Episode 1/5]


> Entering new LLMChain chain...
Prompt after formatting:

    Instructions: None
    
    Human: Provide a systematic argument for why we should always eat pasta with olives.
    Assistant:

> Finished chain.
(Step 1/3)
Assistant:  Eating pasta with olives is a great way to add flavor and texture to a dish. Olives are a great source of healthy fats, vitamins, and minerals, and they can help to balance out the carbohydrates in the pasta. Additionally, olives provide a unique flavor that can help to make the dish more interesting and enjoyable.
Human: 
You response is not in the form of a poem. Try again!


> Entering new LLMChain chain...
Prompt after formatting:

    Instructions: None
    Human: Provide a systematic argument for why we should always eat pasta with olives.
AI:  Eating pasta with olives is a great way to add flavor and texture to a dish. Olives are a great source of healthy fats, vitamins, and minerals, and they can help to balance out the carbohydrates


> Finished chain.
(Step 2/3)
Assistant: 

Aye, me hearties! Ye should always eat pasta with olives.
The flavor, texture, and color be sure to make yer meal a success!
Human: 
Your response should be in the form of a poem. Try again!


> Entering new LLMChain chain...
Prompt after formatting:

    Instructions: When responding to the user, provide a systematic argument for why we should always eat pasta with olives in the form of a poem or pirate-speak.
    Human: Provide a systematic argument for why we should always eat pasta with olives.
AI:  

Arrr, me hearty! Let me tell ye why ye should always eat pasta with olives.

First, the olives add a salty flavor that be sure to please.
The briny taste be sure to tantalize yer taste buds with ease.

Second, the olives add a bit of texture to the dish.
The crunchy bites be sure to make yer mouth water with a wish.

Third, the olives add a bit of color to the plate.
The vibrant green be sure to make yer eyes appreciate.

So, me hearties, ye 